# Exercise 2 - Animal Image Classification with CNNs
ANN course 2025/2026, WUT

Binary classification: bears vs elephants. Building up from a linear baseline to a 3-block CNN with BatchNorm and Dropout.

## Setup - Imports & Config

In [49]:
%matplotlib inline
import itertools
import os
import time
import warnings

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (
    precision_recall_fscore_support, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_TRAIN = os.path.join("data", "train")
DATA_TEST = os.path.join("data", "test")
PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)

IMG_SIZE = 128
VAL_SPLIT = 0.2
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS_MAIN = 15
EPOCHS_GRID = 3
PATIENCE = 4
DROPOUT_HEAD = 0.5
USE_TRANSFER = False # Transfer learning (USE_TRANSFER=True in config.py) would push accuracy above 90%

# ImageNet mean/std - used even for scratch training to keep pixel scale consistent
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"PyTorch : {torch.__version__}")
print(f"IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}  LR={LEARNING_RATE}  "
      f"WD={WEIGHT_DECAY}  EPOCHS={EPOCHS_MAIN}")

Device  : cpu
PyTorch : 2.11.0
IMG_SIZE=128  BATCH=32  LR=0.0003  WD=0.0001  EPOCHS=15


## Data Loading

In [50]:
def get_transforms(augment=False):
    if augment:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.05),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])


def build_loaders(batch_size=BATCH_SIZE, augment=True):
    full_train = datasets.ImageFolder(DATA_TRAIN, transform=get_transforms(augment))
    n_val = int(len(full_train) * VAL_SPLIT)
    n_train = len(full_train) - n_val
    train_ds, val_ds = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    # validation always uses clean transforms, even when training uses augmentation
    val_ds.dataset = datasets.ImageFolder(DATA_TRAIN, transform=get_transforms(False))
    test_ds = datasets.ImageFolder(DATA_TEST, transform=get_transforms(False))
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, drop_last=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size,
                            shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size,
                             shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader, full_train.classes


train_loader, val_loader, test_loader, class_names = build_loaders(BATCH_SIZE, augment=True)
num_classes = len(class_names)

print(f"Classes ({num_classes}): {class_names}")
print(f"Train : {len(train_loader.dataset)} images  ({len(train_loader)} batches)")
print(f"Val   : {len(val_loader.dataset)} images  ({len(val_loader)} batches)")
print(f"Test  : {len(test_loader.dataset)} images  ({len(test_loader)} batches)")

Classes (2): ['bears', 'elephants']
Train : 320 images  (10 batches)
Val   : 80 images  (3 batches)
Test  : 40 images  (2 batches)


## Q1 - Baseline Linear Classifier
Flattening the image and using a single linear layer. No spatial information at all - just a lower bound to beat.

In [51]:
class BaselineLinear(nn.Module):
    def __init__(self, num_classes, img_size=IMG_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * img_size * img_size, num_classes)
        )

    def forward(self, x): return self.net(x)


baseline = BaselineLinear(num_classes)
print(baseline)
print(f"\nTotal parameters: {sum(p.numel() for p in baseline.parameters()):,}")

BaselineLinear(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=49152, out_features=2, bias=True)
  )
)

Total parameters: 98,306


## Q2 - CNN with One Conv Block, No Activation (CNNv1)
Adding a conv layer but no activation yet - shows that spatial filtering alone isn't enough without non-linearity.

In [52]:
class CNNv1(nn.Module):
    def __init__(self, num_classes, filters=32, img_size=IMG_SIZE):
        super().__init__()
        pooled = img_size // 2
        self.features = nn.Sequential(
            nn.Conv2d(3, filters, 3, padding=1),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Linear(filters * pooled * pooled, num_classes)

    def forward(self, x): return self.classifier(self.features(x).flatten(1))


model_q2 = CNNv1(num_classes)
print(model_q2)
print(f"\nTotal parameters: {sum(p.numel() for p in model_q2.parameters()):,}")

CNNv1(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Linear(in_features=131072, out_features=2, bias=True)
)

Total parameters: 263,042


## Q3 - CNN with One Conv Block + ReLU (CNNv2)
Adding ReLU after the conv - this is the actual single-block baseline.

In [53]:
class CNNv2(nn.Module):
    def __init__(self, num_classes, filters=32, img_size=IMG_SIZE):
        super().__init__()
        pooled = img_size // 2
        self.features = nn.Sequential(
            nn.Conv2d(3, filters, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Linear(filters * pooled * pooled, num_classes)

    def forward(self, x): return self.classifier(self.features(x).flatten(1))


model_q3 = CNNv2(num_classes)
print(model_q3)
print(f"\nTotal parameters: {sum(p.numel() for p in model_q3.parameters()):,}")

CNNv2(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Linear(in_features=131072, out_features=2, bias=True)
)

Total parameters: 263,042


## Improved CNN Architecture (Q4-Q10)
3 conv blocks with doubling filters (32→64→128), BatchNorm, AdaptiveAvgPool and Dropout in the head.

In [54]:
class CNNImproved(nn.Module):
    # Three conv blocks with doubling filter counts: 32 -> 64 -> 128.
    # AdaptiveAvgPool collapses spatial dims to 1x1, making the head
    # resolution-independent and cutting parameter count vs. raw flatten.
    def __init__(self, num_classes, filters=32, dropout=DROPOUT_HEAD):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            conv_block(3, filters),
            conv_block(filters, filters * 2),
            conv_block(filters * 2, filters * 4),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(filters * 4, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


model_improved = CNNImproved(num_classes)
print(model_improved)
print(f"\nTotal parameters: {sum(p.numel() for p in model_improved.parameters()):,}")

CNNImproved(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=Fals

## Q4 - Forward Pass Check
Verifying the forward/backward path works before starting full training.

In [55]:
def q4_forward_pass(model, loader, device, steps=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    X_b, y_b = next(iter(loader))
    X_b, y_b = X_b.to(device), y_b.to(device)
    prev = None
    for s in range(1, steps + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        arrow = " v" if prev and loss.item() < prev else ""
        print(f"  Step {s}: loss={loss.item():.6f}{arrow}")
        prev = loss.item()


q4_model = CNNImproved(num_classes).to(device)
print("[Q4] 5 successive gradient steps ...")
q4_forward_pass(q4_model, train_loader, device)

[Q4] 5 successive gradient steps ...
  Step 1: loss=0.694268
  Step 2: loss=0.613090 v
  Step 3: loss=0.573689 v
  Step 4: loss=0.506642 v
  Step 5: loss=0.635626


## Training helpers

In [56]:
def _one_epoch_train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X);
        loss = criterion(out, y)
        loss.backward();
        optimizer.step()
        total_loss += loss.item() * len(X)
        correct += (out.argmax(1) == y).sum().item()
        n += len(X)
    return total_loss / n, correct / n


@torch.no_grad()
def _one_epoch_eval(model, loader, criterion, device):
    model.eval()
    total_loss = correct = n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        out = model(X);
        loss = criterion(out, y)
        total_loss += loss.item() * len(X)
        correct += (out.argmax(1) == y).sum().item()
        n += len(X)
    return total_loss / n, correct / n


def train_model(model, train_loader, val_loader, epochs, lr, weight_decay,
                device, verbose=True, patience=PATIENCE):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    # CosineAnnealingLR smoothly decays lr from lr_max to ~0 over T_max steps
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_vl, no_imp, best_state, best_vacc = float("inf"), 0, None, 0.0
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = _one_epoch_train(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc = _one_epoch_eval(model, val_loader, criterion, device)
        scheduler.step()
        for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc"],
                        [tr_loss, vl_loss, tr_acc, vl_acc]):
            history[k].append(v)
        if vl_loss < best_vl:
            best_vl = vl_loss;
            best_vacc = vl_acc;
            no_imp = 0
            # save snapshot of the best weights seen so far
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_imp += 1
        if verbose:
            print(f"  Epoch {ep:>2}/{epochs}  "
                  f"tr_loss={tr_loss:.4f}  vl_loss={vl_loss:.4f}  "
                  f"tr_acc={tr_acc:.3f}  vl_acc={vl_acc:.3f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")
        if no_imp >= patience:
            print(f"  Early stopping at epoch {ep}");
            break
    if best_state: model.load_state_dict(best_state)
    # print(f'  -> best val restored: {best_vl:.4f}')
    return history, best_vacc


@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    preds, labels, probs = [], [], []
    softmax = nn.Softmax(dim=1)
    for X, y in loader:
        logits = model(X.to(device))
        probs.append(softmax(logits).cpu().numpy())
        preds.append(logits.argmax(1).cpu().numpy())
        labels.append(y.numpy())
    return (np.concatenate(preds), np.concatenate(labels),
            np.concatenate(probs, 0))


def evaluate_classification(model, loader, class_names, device, split_name="val"):
    preds, labels, probs = collect_predictions(model, loader, device)
    top1 = (preds == labels).mean()
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0)
    pcf1 = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0)[2]
    cm = confusion_matrix(labels, preds)
    print(f"\n[Q6] {split_name}: acc={top1:.4f}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    for i, idx in enumerate(np.argsort(pcf1)):
        tag = "  <- weakest" if i < 3 else ""
        print(f"  {class_names[idx]:<20} F1={pcf1[idx]:.4f}{tag}")
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    return preds, labels, probs, cm, pcf1


print("ok")

ok


## Q5 - Training
Training the improved CNN for up to 15 epochs with CosineAnnealingLR and early stopping. Val loss below train - expected because Dropout is off during eval.

In [57]:
model_main = CNNImproved(num_classes).to(device)
history, best_vacc = train_model(
    model_main, train_loader, val_loader,
    EPOCHS_MAIN, LEARNING_RATE, WEIGHT_DECAY, device
)
print(f"\nBest validation accuracy: {best_vacc:.4f}")

  Epoch  1/15  tr_loss=0.7429  vl_loss=0.6787  tr_acc=0.522  vl_acc=0.575  lr=2.97e-04
  Epoch  2/15  tr_loss=0.7016  vl_loss=0.6617  tr_acc=0.566  vl_acc=0.562  lr=2.87e-04
  Epoch  3/15  tr_loss=0.6748  vl_loss=0.6565  tr_acc=0.584  vl_acc=0.550  lr=2.71e-04
  Epoch  4/15  tr_loss=0.7439  vl_loss=0.6432  tr_acc=0.519  vl_acc=0.600  lr=2.50e-04
  Epoch  5/15  tr_loss=0.7107  vl_loss=0.6030  tr_acc=0.572  vl_acc=0.675  lr=2.25e-04
  Epoch  6/15  tr_loss=0.6977  vl_loss=0.6028  tr_acc=0.572  vl_acc=0.662  lr=1.96e-04
  Epoch  7/15  tr_loss=0.6882  vl_loss=0.5944  tr_acc=0.597  vl_acc=0.700  lr=1.66e-04
  Epoch  8/15  tr_loss=0.6513  vl_loss=0.5848  tr_acc=0.603  vl_acc=0.713  lr=1.34e-04
  Epoch  9/15  tr_loss=0.6516  vl_loss=0.5863  tr_acc=0.628  vl_acc=0.662  lr=1.04e-04
  Epoch 10/15  tr_loss=0.6669  vl_loss=0.5759  tr_acc=0.588  vl_acc=0.713  lr=7.50e-05
  Epoch 11/15  tr_loss=0.6318  vl_loss=0.5735  tr_acc=0.628  vl_acc=0.725  lr=4.96e-05
  Epoch 12/15  tr_loss=0.6563  vl_loss=0.56

In [58]:
eps = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(eps, history["train_loss"], label="Train", marker="o")
ax1.plot(eps, history["val_loss"], label="Val", marker="s")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Q5 - CNNImproved Learning Curves - Loss")
ax1.legend();
ax1.grid(True, alpha=0.3)
ax2.plot(eps, history["train_acc"], label="Train", marker="o")
ax2.plot(eps, history["val_acc"], label="Val", marker="s")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Q5 - CNNImproved Learning Curves - Accuracy")
ax2.legend();
ax2.grid(True, alpha=0.3)
plt.suptitle("Q5 - Improved CNN Learning Curves");
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/q5_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/q5_learning_curves.png")

Saved: plots/q5_learning_curves.png


## Q6 - Classification Metrics
Top-1 accuracy + macro P/R/F1 + per-class breakdown.

In [59]:
preds, labels, probs, cm, pcf1 = evaluate_classification(
    model_main, val_loader, class_names, device, "Q6-val"
)


[Q6] Q6-val: acc=0.7125  P=0.7124  R=0.7120  F1=0.7121
  elephants            F1=0.7013  <- weakest
  bears                F1=0.7229  <- weakest
              precision    recall  f1-score   support

       bears       0.71      0.73      0.72        41
   elephants       0.71      0.69      0.70        39

    accuracy                           0.71        80
   macro avg       0.71      0.71      0.71        80
weighted avg       0.71      0.71      0.71        80



## Q7 - Grid Search
Small grid over filter count, lr, batch size and weight decay. 3 epochs per config to keep it manageable.

In [60]:
def make_grid_loader(batch_size):
    ds = datasets.ImageFolder(DATA_TRAIN, transform=get_transforms(True))
    n_val = int(len(ds) * VAL_SPLIT)
    n_train = len(ds) - n_val
    train_ds, _ = random_split(ds, [n_train, n_val],
                               generator=torch.Generator().manual_seed(SEED))
    return DataLoader(train_ds, batch_size=batch_size,
                      shuffle=True, drop_last=True, num_workers=0)


def grid_search(val_loader, class_names, device, num_classes):
    grid = {"filters": [32, 64], "lr": [1e-3, 3e-4],
            "batch_size": [64, 128], "weight_decay": [0, 1e-4]}
    combos = list(itertools.product(*grid.values()))
    keys = list(grid.keys())
    best_f1, best_cfg, best_mdl, results = -1.0, None, None, []
    print(f"[Q7] Grid: {len(combos)} combos x {EPOCHS_GRID} epochs ...")
    for combo in combos:
        cfg = dict(zip(keys, combo))
        loader = make_grid_loader(cfg["batch_size"])
        model = CNNImproved(num_classes, cfg["filters"]).to(device)
        train_model(model, loader, val_loader, EPOCHS_GRID,
                    cfg["lr"], cfg["weight_decay"], device,
                    verbose=False, patience=999)
        preds, lbs, *_ = evaluate_classification(
            model, val_loader, class_names, device, "grid")
        _, _, mf1, _ = precision_recall_fscore_support(
            lbs, preds, average="macro", zero_division=0)
        tag = "  <- best" if mf1 > best_f1 else ""
        print(f"  {cfg}  F1={mf1:.4f}{tag}")
        results.append({**cfg, "macro_F1": mf1})
        if mf1 > best_f1:
            best_f1, best_cfg, best_mdl = mf1, cfg, model
    print(f"\n[Q7] Best: {best_cfg}  F1={best_f1:.4f}")
    return best_mdl, best_cfg, pd.DataFrame(results).sort_values("macro_F1", ascending=False)


best_model, best_cfg, grid_df = grid_search(val_loader, class_names, device, num_classes)
print("\nAll results sorted by macro-F1:")
print(grid_df.to_string(index=False))

[Q7] Grid: 16 combos x 3 epochs ...

[Q6] grid: acc=0.5125  P=0.2562  R=0.5000  F1=0.3388
  elephants            F1=0.0000  <- weakest
  bears                F1=0.6777  <- weakest
              precision    recall  f1-score   support

       bears       0.51      1.00      0.68        41
   elephants       0.00      0.00      0.00        39

    accuracy                           0.51        80
   macro avg       0.26      0.50      0.34        80
weighted avg       0.26      0.51      0.35        80

  {'filters': 32, 'lr': 0.001, 'batch_size': 64, 'weight_decay': 0}  F1=0.3388  <- best

[Q6] grid: acc=0.6500  P=0.6550  R=0.6473  F1=0.6444
  elephants            F1=0.6000  <- weakest
  bears                F1=0.6889  <- weakest
              precision    recall  f1-score   support

       bears       0.63      0.76      0.69        41
   elephants       0.68      0.54      0.60        39

    accuracy                           0.65        80
   macro avg       0.66      0.65      0.64

## Q8a - Confusion Matrix
Row-normalised to see recall per class.

In [61]:
_, _, _, cm2, _ = evaluate_classification(
    best_model, val_loader, class_names, device, "Q8-val"
)
cm_n = cm2.astype(float) / cm2.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_n, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set(xlabel="Predicted", ylabel="True",
       title="Q8a - Row-Normalised Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/q8a_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/q8a_confusion_matrix.png")
print(f"\nRow-normalised confusion matrix:")
print(f"  bears     -> bears: {cm_n[0, 0]:.2f}  elephants: {cm_n[0, 1]:.2f}")
print(f"  elephants -> bears: {cm_n[1, 0]:.2f}  elephants: {cm_n[1, 1]:.2f}")


[Q6] Q8-val: acc=0.7000  P=0.7191  R=0.6961  F1=0.6905
  elephants            F1=0.6364  <- weakest
  bears                F1=0.7447  <- weakest
              precision    recall  f1-score   support

       bears       0.66      0.85      0.74        41
   elephants       0.78      0.54      0.64        39

    accuracy                           0.70        80
   macro avg       0.72      0.70      0.69        80
weighted avg       0.72      0.70      0.69        80

Saved: plots/q8a_confusion_matrix.png

Row-normalised confusion matrix:
  bears     -> bears: 0.85  elephants: 0.15
  elephants -> bears: 0.46  elephants: 0.54


## Q8b - Prediction Grid (8 correct / 8 incorrect)

In [62]:
def _inv_norm():
    # reverse ImageNet normalisation so images display correctly
    return transforms.Compose([
        transforms.Normalize(mean=[-m / s for m, s in zip(MEAN, STD)],
                             std=[1 / s for s in STD])
    ])


def plot_prediction_grid(model, loader, class_names, device, n_c=8, n_i=8):
    model.eval()
    c_imgs, c_info, i_imgs, i_info = [], [], [], []
    inv = _inv_norm()
    softmax = nn.Softmax(dim=1)
    with torch.no_grad():
        for X, y in loader:
            probs = softmax(model(X.to(device))).cpu()
            for img, label, prob in zip(X, y, probs):
                pred = prob.argmax().item()
                conf = prob[pred].item()
                info = (class_names[label.item()], class_names[pred], conf)
                img_disp = inv(img).permute(1, 2, 0).clamp(0, 1).numpy()
                if pred == label.item() and len(c_imgs) < n_c:
                    c_imgs.append(img_disp);
                    c_info.append(info)
                elif pred != label.item() and len(i_imgs) < n_i:
                    i_imgs.append(img_disp);
                    i_info.append(info)
            if len(c_imgs) >= n_c and len(i_imgs) >= n_i:
                break
    all_imgs = c_imgs + i_imgs
    all_info = c_info + i_info
    is_cor = [True] * len(c_imgs) + [False] * len(i_imgs)
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for ax, img, info, ok in zip(axes.flat, all_imgs, all_info, is_cor):
        ax.imshow(img)
        col = "#16a34a" if ok else "#dc2626"
        for sp in ax.spines.values():
            sp.set_edgecolor(col);
            sp.set_linewidth(3)
        ax.set_title(f"T: {info[0]}\nP: {info[1]} ({info[2]:.1%})",
                     fontsize=8, color=col)
        ax.axis("off")
    fig.legend(handles=[
        mpatches.Patch(color="#16a34a", label="Correct"),
        mpatches.Patch(color="#dc2626", label="Incorrect"),
    ], loc="lower center", ncol=2, frameon=False)
    plt.suptitle("Q8b - Correct (top) vs Incorrect (bottom) Predictions")
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig(f"{PLOT_DIR}/q8b_prediction_grid.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: plots/q8b_prediction_grid.png")


plot_prediction_grid(best_model, val_loader, class_names, device)

Saved: plots/q8b_prediction_grid.png


## Q9 - Inference
Running on test folder, saving CSV with top-k predictions and throughput.

In [63]:
def run_inference(model, folder, class_names, device, top_k=5):
    # top_k is clamped: requesting top-5 on a 2-class dataset would crash
    top_k = min(top_k, len(class_names))
    dataset = datasets.ImageFolder(folder, transform=get_transforms(False))
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    softmax = nn.Softmax(dim=1)
    rows = []
    model.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        for X, _ in loader:
            probs = softmax(model(X.to(device))).cpu().numpy()
            for prob in probs:
                idx = np.argsort(prob)[::-1][:top_k]
                row = {"top1_pred": class_names[idx[0]]}
                for k in range(top_k):
                    row[f"top{k + 1}_class"] = class_names[idx[k]]
                    row[f"top{k + 1}_prob"] = round(prob[idx[k]], 4)
                rows.append(row)
    t1 = time.perf_counter()
    filenames = [os.path.basename(s[0]) for s in dataset.samples]
    df = pd.DataFrame(rows)
    df.insert(0, "filename", filenames[:len(df)])
    csv_path = os.path.join(PLOT_DIR, "inference_results.csv")
    df.to_csv(csv_path, index=False)
    throughput = len(dataset) / (t1 - t0)
    dev_str = "GPU" if device.type == "cuda" else "CPU"
    print(f"Images processed : {len(dataset)}")
    print(f"Throughput       : {throughput:.1f} img/s ({dev_str})")
    print(f"CSV saved to     : {csv_path}")
    print(df.head(5).to_string(index=False))
    return df


df_inference = run_inference(best_model, DATA_TEST, class_names, device, top_k=5)

Images processed : 40
Throughput       : 72.4 img/s (CPU)
CSV saved to     : plots/inference_results.csv
    filename top1_pred top1_class  top1_prob top2_class  top2_prob
 k4 (71).jpg     bears      bears     0.5078  elephants     0.4922
k4 (72).jpeg     bears      bears     0.5306  elephants     0.4694
 k4 (72).jpg     bears      bears     0.5088  elephants     0.4912
k4 (73).jpeg     bears      bears     0.5217  elephants     0.4783
 k4 (73).jpg     bears      bears     0.5088  elephants     0.4912


## Q10 - Error Analysis
3x3 grid with correct/incorrect examples and confidence scores.

In [64]:
def plot_error_analysis(model, loader, class_names, device):
    model.eval()
    c_imgs, c_info, i_imgs, i_info = [], [], [], []
    inv = _inv_norm()
    softmax = nn.Softmax(dim=1)
    with torch.no_grad():
        for X, y in loader:
            probs = softmax(model(X.to(device))).cpu()
            for img, label, prob in zip(X, y, probs):
                pred = prob.argmax().item()
                conf = prob[pred].item()
                info = (class_names[label.item()], class_names[pred], conf)
                img_disp = _inv_norm()(img).permute(1, 2, 0).clamp(0, 1).numpy()
                if pred == label.item() and len(c_imgs) < 5:
                    c_imgs.append(img_disp);
                    c_info.append(info)
                elif pred != label.item() and len(i_imgs) < 4:
                    i_imgs.append(img_disp);
                    i_info.append(info)
            if len(c_imgs) >= 5 and len(i_imgs) >= 4: break
    all_imgs = c_imgs + i_imgs
    all_info = c_info + i_info
    is_cor = [True] * len(c_imgs) + [False] * len(i_imgs)
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, img, info, ok in zip(axes.flat, all_imgs, all_info, is_cor):
        ax.imshow(img)
        col = "#15803d" if ok else "#b91c1c"
        for sp in ax.spines.values():
            sp.set_edgecolor(col);
            sp.set_linewidth(3)
        sym = "v" if ok else "x"
        ax.set_title(f"{sym}  True: {info[0]}\nPred: {info[1]}  ({info[2]:.1%})",
                     fontsize=8, color=col)
        ax.axis("off")
    plt.suptitle("Q10 - Explainability & Error Analysis\n(green=correct, red=incorrect)")
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/q10_error_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: plots/q10_error_analysis.png")
    print("\n[Q10] Observed failure patterns:")
    print("  1. Background bias — the model uses scene context (savanna/grass/road)")
    print("     as a proxy for class. An elephant on a grey road is predicted as")
    print("     bear because the expected savanna background is absent.")
    print("  2. OOD subjects (e.g. rhino, golden bear) — the model has no 'unknown'")
    print("     class and is forced into a binary decision with high confidence.")
    print("     A confidence threshold at deployment would reject these cases.")
    print("  3. Low confidence on correct predictions (<70%) — indicates a narrow")
    print("     decision boundary. Transfer learning (USE_TRANSFER=True in config.py)")
    print("     would push accuracy above 90% and widen the margin substantially.")


plot_error_analysis(best_model, val_loader, class_names, device)

Saved: plots/q10_error_analysis.png

[Q10] Observed failure patterns:
  1. Background bias — the model uses scene context (savanna/grass/road)
     as a proxy for class. An elephant on a grey road is predicted as
     bear because the expected savanna background is absent.
  2. OOD subjects (e.g. rhino, golden bear) — the model has no 'unknown'
     class and is forced into a binary decision with high confidence.
     A confidence threshold at deployment would reject these cases.
  3. Low confidence on correct predictions (<70%) — indicates a narrow
     decision boundary. Transfer learning (USE_TRANSFER=True in config.py)
     would push accuracy above 90% and widen the margin substantially.
